# Lake 16-Tishreen: a concise method comparison

**Question.** Does a Landsat-derived NDWI estimate reproduce the September water-area decline reported in the MSc thesis?

**Design.** Compare the 11 Tishreen image-years with acquisition dates listed in thesis Table 2: 2002, 2004–2007, 2012, and 2015, 2017–2020. GEE values are year-level September composites, not the same individual scenes. September is retained because the thesis selected it as a post-summer, post-drought observation window (§2.2.4). Thesis areas came from supervised object-based classification with visual correction; the GEE series uses September monthly-median Landsat composites and NDWI > 0.15. Python reports paired differences, OLS slopes with 95% confidence intervals, Sen slopes, and exact Mann–Kendall p-values.

**Interpretation and limits.** The series use different sensors, acquisition dates, resolutions, and classification procedures; NDWI > 0.15 is the prior workflow's primary threshold, not a threshold calibrated against independent observations. This is a method-comparability analysis, not independent validation. It cannot attribute lake changes to climate or dam operation. The hand-drawn ROI geometry was not preserved, so this notebook reproduces statistics from saved estimates, not image extraction. Trend p-values are exploratory and do not correct for serial dependence.

## Abstract (draft)

This exploratory extension evaluates whether the September surface-water trend reported for Lake 16-Tishreen can be reproduced using Landsat-derived NDWI estimates exported from Google Earth Engine. Eleven years with Tishreen image dates documented in thesis Table 2 (2002, 2004–2007, 2012, 2015, and 2017–2020) were compared with year-level GEE estimates. September was retained because the MSc thesis selected it to represent conditions after summer and the dry season. Thesis areas, produced by supervised object-based classification with visual correction, were compared with monthly-median Landsat estimates using NDWI > 0.15. The thesis series showed a declining trend (OLS slope −0.125 km²/year, p = 0.0049; Mann–Kendall p = 0.0165), whereas the NDWI series showed no significant linear or monotonic trend (−0.008 km²/year, p = 0.712; Mann–Kendall p = 0.761). NDWI estimates averaged 11.8% below thesis values; the mean difference was larger in 2002–2007 (−19.2%) than in 2012–2020 (−5.5%). Because sensors, dates, spatial resolution, and mapping methods differ, this comparison evaluates method dependence rather than accuracy or hydrological causation. The original ROI geometry was unavailable, so satellite-area extraction is not repeated here.

**CV wording:** “Used Python to compare Google Earth Engine-derived Landsat NDWI water-area estimates with thesis-reported classifications for Lake 16-Tishreen across 11 thesis-dated September years; quantified method-dependent trend differences using OLS, Sen slope, and Mann–Kendall statistics.”

This wording does not claim that Python reprocessed the satellite images.

## Data provenance

- Thesis values and image dates: Salloum (2021), Table 2 and Table 5; September rationale: §2.2.4. Table 5 also reports 2016 (5.61 km²), but Table 2 lists no Tishreen image date for that year; 2016 is excluded from the primary paired analysis.
- GEE estimates: Tishreen_September_2002_2020_QC.csv, column area_NDWI_015_km2, from the earlier Google Earth Engine export.
- NDWI reference: McFeeters, S. K. (1996). The use of the Normalized Difference Water Index (NDWI) in the delineation of open water features. International Journal of Remote Sensing, 17(7), 1425–1432. doi:10.1080/01431169608948714.
- Thesis reference: Salloum, M. (2021). Climate Change Consequences and Scenarios: Syria as a Case Study. MSc thesis, University of Tübingen.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress, kendalltau, theilslopes, t

# Matched-year values: thesis Table 5 and the earlier GEE export
df = pd.DataFrame({
    "year": [2002, 2004, 2005, 2006, 2007, 2012, 2015, 2017, 2018, 2019, 2020],
    "thesis_km2": [9.84, 8.23, 9.74, 8.37, 7.30, 7.46, 6.86, 7.20, 7.71, 7.47, 6.55],
    "gee_ndwi_015_km2": [
        7.5628113505354335, 6.990907569258148, 7.0042640296358,
        6.782836239633358, 6.51245589544223, 6.00847987093817,
        6.618140335623229, 7.048176741282743, 7.409241146879785,
        7.086906545288803, 6.607902218232279
    ]
})
df["difference_pct"] = 100 * (df["gee_ndwi_015_km2"] - df["thesis_km2"]) / df["thesis_km2"]

def trend(label, column):
    x, y = df["year"].to_numpy(), df[column].to_numpy()
    fit = linregress(x, y)
    margin = t.ppf(0.975, len(y) - 2) * fit.stderr
    sen = theilslopes(y, x, alpha=0.95)
    mk = kendalltau(x, y, method="exact")
    return {
        "series": label,
        "n": len(y),
        "OLS slope (km²/year)": fit.slope,
        "OLS 95% CI low": fit.slope - margin,
        "OLS 95% CI high": fit.slope + margin,
        "OLS p": fit.pvalue,
        "Sen slope (km²/year)": sen.slope,
        "Mann–Kendall tau": mk.statistic,
        "Mann–Kendall p (exact)": mk.pvalue
    }

summary = pd.DataFrame([
    trend("Thesis classification", "thesis_km2"),
    trend("GEE Landsat NDWI > 0.15", "gee_ndwi_015_km2")
])
display(summary.round(4))
display(df.round(2))

print("Mean NDWI-minus-thesis difference: {:.1f}%".format(df["difference_pct"].mean()))
print("2002–2007 mean difference: {:.1f}%".format(df.loc[df.year <= 2007, "difference_pct"].mean()))
print("2012–2020 mean difference: {:.1f}%".format(df.loc[df.year >= 2012, "difference_pct"].mean()))
print("Observed years only; no interpolation between missing thesis image-years.")

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.scatter(df["year"], df["thesis_km2"], marker="s", s=55, label="Thesis: object-based classification")
ax.scatter(df["year"], df["gee_ndwi_015_km2"], marker="o", s=55, label="GEE: Landsat NDWI > 0.15")
ax.set(xlabel="Year", ylabel="Mapped surface-water area (km²)",
       title="Lake 16-Tishreen: September estimates in thesis-dated years")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig("tishreen_method_comparison.png", dpi=300, bbox_inches="tight")
df.to_csv("tishreen_paired_areas.csv", index=False)
plt.show()
print("Saved: tishreen_method_comparison.png and tishreen_paired_areas.csv")
